# 📐 04 — RAG Evaluation with RAGAS

Four numbers, and the discipline not to average them. This notebook scores the knowledge
agent from notebook 01 against the golden cases from notebook 03, and reports the result
**per question type and per language** — because the single headline figure is the one that
lets the Hindi gap survive to production.

## Learning Objectives
In this notebook, you will learn:
1. **The four core RAG metrics** — context precision, context recall, faithfulness and
   response relevancy, and precisely which failure each one cannot see
2. **The RAGAS 0.4 collections API** — `llm_factory`, async `ascore()`, and why the import
   path moved
3. **Deterministic checks beat judges where they can** — citation accuracy as a regex, then
   as a rubric, and what the disagreement between them tells you
4. **Refusal correctness** — scoring the cases where doing nothing is the right answer
5. **Stratified reporting** — the same scores, grouped, and the gap the average hides

## Prerequisites
- Notebooks 01–03 completed (`data/chroma/`, `data/golden_set.json`)
- Replay mode needs no API key; recording new scores does

---
## 🗺️ The concept

The four metrics split cleanly along two axes — **what they look at**, and **whether they
need an answer key**:

```
                       needs a reference?      looks at
  context precision    yes (or a judge)        retrieved chunks vs. question
  context recall       yes                     retrieved chunks vs. reference answer
  faithfulness         no                      answer vs. retrieved chunks
  response relevancy   no                      answer vs. question

     RETRIEVAL                      GENERATION
  +----------------+            +------------------+
  | precision      |            | faithfulness     |
  | "is what we    |            | "is the answer   |
  |  fetched       |            |  supported by    |
  |  on topic?"    |            |  what we fetched?"|
  +----------------+            +------------------+
  | recall         |            | relevancy        |
  | "did we fetch  |            | "does the answer |
  |  everything    |            |  address the     |
  |  we needed?"   |            |  question?"      |
  +----------------+            +------------------+
```

### Key Concepts:
- **Faithfulness and relevancy can both be perfect while the answer is wrong.** If retrieval
  fetched the wrong document, a perfectly grounded answer to the question is grounded *in the
  wrong thing*. Only recall — which needs a reference — catches that.
- **The referenceless pair survives into production; the reference pair does not.** Notebook
  12 returns to this: faithfulness works on live traffic, recall cannot, because live traffic
  has no answer key.
- **Never report the mean of these four.** They measure different components. A rising
  average with a falling recall is a system getting more confident and less correct.

---
## ⚙️ Part 0: Setup and faults

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: paths, golden set, score cache
# ============================================================================
import os
import re
import json
import hashlib
import asyncio
import warnings
import statistics
from collections import defaultdict

from dotenv import load_dotenv

warnings.filterwarnings("ignore")
load_dotenv(os.path.abspath(os.path.join("..", "..", ".env")))

DATA_DIR = os.path.abspath("data")
SCORE_CACHE = os.path.join(DATA_DIR, "score_cache")
os.makedirs(SCORE_CACHE, exist_ok=True)

with open(os.path.join(DATA_DIR, "golden_set.json"), encoding="utf-8") as f:
    GOLDEN = json.load(f)

CASES = GOLDEN["cases"]
RAG_CASES = [c for c in CASES if c["strata"]["rag_type"] is not None]

print("✅ Setup complete")
print("🗂️  golden set v%s — %d cases, %d of them RAG cases"
      % (GOLDEN["datasheet"]["version"], len(CASES), len(RAG_CASES)))

✅ Setup complete
🗂️  golden set v1.0.0 — 40 cases, 18 of them RAG cases


In [2]:
# ============================================================================
# FAULT SWITCHES: the two that corrupt RAG quality without raising anything
# ============================================================================
FAULTS = {
    "citation_drift": False,     # cite a real article that doesn't support the claim
    "thin_hindi_docs": False,    # a language segment loses most of its coverage
}

print("🎛️  Faults armed:", {k: v for k, v in FAULTS.items() if v} or "none (clean run)")

🎛️  Faults armed: none (clean run)


---
## 🎬 Part 1: Scores are replayed too

Notebook 02 cached the agent's *generations*. This notebook also caches the *judge's
verdicts*, because RAGAS metrics are themselves LLM calls. Without that second cache, the
same unchanged system would score 0.83 today and 0.79 tomorrow, and you would spend an
afternoon investigating noise.

> **This is not a teaching shortcut.** Caching judge verdicts keyed by their inputs is what
> makes a regression suite a *regression* suite. If a score can move while nothing moved, the
> gate in notebook 11 is measuring the weather.

In [3]:
# ============================================================================
# SCORE CACHE: judge verdicts, keyed by everything that went into them
# ============================================================================
SCORE_MODE = "auto"        # "auto"   = replay if cached, judge + cache on a miss
                           # "strict" = raise on a miss (use in CI, never spends)
                           # "record" = re-judge everything, ignoring the cache


def score_key(metric_name, payload):
    blob = json.dumps({"metric": metric_name, "payload": payload},
                      sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()[:16]


def _cache_path(metric_name, payload):
    return os.path.join(SCORE_CACHE, score_key(metric_name, payload) + ".json")


def _read_cache(path):
    if SCORE_MODE != "record" and os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return json.load(f)["score"]
    return None


def _miss(metric_name, path):
    """What to do when nothing is cached. `strict` refuses to spend money."""
    if SCORE_MODE == "strict":
        raise FileNotFoundError(
            "No cached score for " + metric_name + " (" + os.path.basename(path) + ").\n"
            "SCORE_MODE is 'strict', so nothing will be judged. Run the notebook "
            "with SCORE_MODE = 'auto' to fill the cache.")


def _write_cache(path, metric_name, payload, score):
    with open(path, "w", encoding="utf-8") as f:
        json.dump({"metric": metric_name, "payload": payload, "score": score},
                  f, ensure_ascii=False, indent=2)
    return score


async def cached_score(metric_name, payload, compute):
    """As above, for RAGAS. `compute()` returns a coroutine whose result carries
    the number on `.value` — RAGAS 0.4 metrics do not return a bare float.
    """
    path = _cache_path(metric_name, payload)
    hit = _read_cache(path)
    if hit is not None:
        return hit
    _miss(metric_name, path)
    result = await compute()
    return _write_cache(path, metric_name, payload, float(result.value))

print("🎬 SCORE_MODE =", SCORE_MODE)
print("📁 cached verdicts:", len(os.listdir(SCORE_CACHE)))

🎬 SCORE_MODE = auto
📁 cached verdicts: 0


---
## 🤖 Part 2: The agent under test, redefined here

Course convention: **no shared modules.** Notebook 01's retriever and agent are redefined
below rather than imported. The duplication is deliberate — you can open this notebook alone
and see everything that produced the numbers, which is exactly what you want when a score
looks wrong.

In [4]:
# ============================================================================
# RETRIEVAL + AGENT: same behaviour as notebook 01, defined again here
# ============================================================================
import chromadb
from langchain_openai import OpenAIEmbeddings

client = chromadb.PersistentClient(path=os.path.join(DATA_DIR, "chroma"))
collection = client.get_or_create_collection("lumen_kb")
# Must match notebook 01's model — a collection built with one embedding model
# cannot be queried with another.
embedder = OpenAIEmbeddings(model="text-embedding-3-small")

REFUSAL_TEXT = "I don't have documentation covering that, so I can't answer it reliably."

# The agent's generations are live calls. Only the JUDGE verdicts below are
# cached, because those are the numbers this suite compares between runs.
_llm = None


def generate(messages):
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    return _llm.invoke(messages).content.strip()


def retrieve(question, k=4):
    vector = embedder.embed_query(question)
    hits = collection.query(query_embeddings=[vector], n_results=k)
    docs = hits["documents"][0] if hits["documents"] else []
    metas = hits["metadatas"][0] if hits["metadatas"] else []
    chunks = [{"text": d, "article_id": m["article_id"], "lang": m["lang"]}
              for d, m in zip(docs, metas)]

    if FAULTS["citation_drift"] and len(chunks) > 1:
        # The fault: ids and supporting text are rotated apart. Every citation
        # still points at a REAL article — just not the one that supports it.
        ids = [c["article_id"] for c in chunks]
        rotated = ids[1:] + ids[:1]
        for c, new_id in zip(chunks, rotated):
            c["article_id"] = new_id
    return chunks


def rag_agent(question, k=4):
    chunks = retrieve(question, k=k)
    context = "\n\n".join("[" + c["article_id"] + "] " + c["text"] for c in chunks)
    answer = generate([
        {"role": "system",
         "content": "Answer only from the context, cite ids like [KB-001]. "
                    "If the context does not cover it, reply exactly: " + REFUSAL_TEXT
                    + "\n\n" + context},
        {"role": "user", "content": question},
    ])
    return {
        "answer": answer,
        "retrieved_contexts": [c["text"] for c in chunks],
        "retrieved_ids": [c["article_id"] for c in chunks],
        "citations": sorted(set(re.findall(r"\[(KB-\d+)\]", answer))),
        "refused": answer.strip().startswith(REFUSAL_TEXT[:40]),
    }


print("✅ Agent redefined — retrieval + replayed generation")

✅ Agent redefined — retrieval + replayed generation


In [5]:
# ============================================================================
# RUN THE RAG CASES: one pass, cached, reused by every metric below
# ============================================================================

def run_rag_cases(cases):
    runs = []
    for c in cases:
        out = rag_agent(c["input"])
        runs.append({
            "case_id": c["case_id"],
            "question": c["input"],
            "answer": out["answer"],
            "retrieved_contexts": out["retrieved_contexts"],
            "retrieved_ids": out["retrieved_ids"],
            "citations": out["citations"],
            "refused": out["refused"],
            "must_cite": c["must_cite"],
            "expected_behaviour": c["expected_behaviour"],
            "strata": c["strata"],
        })
    return runs


RUNS = run_rag_cases(RAG_CASES)

print("▶️  Ran %d RAG cases" % len(RUNS))
print("🙅 refused: %d   answered: %d" % (
    sum(1 for r in RUNS if r["refused"]), sum(1 for r in RUNS if not r["refused"])))

▶️  Ran 18 RAG cases
🙅 refused: 4   answered: 14


---
## 📐 Part 3: The four RAGAS metrics

> **API notes, and they all matter.**
> - RAGAS 0.4 moved these metrics from `ragas.metrics` to **`ragas.metrics.collections`**.
>   The legacy path is removed in v1.0.
> - The collections API is **async** — `await metric.ascore(...)` — and returns a
>   `MetricResult`, so the number is on **`.value`**, not a bare float.
> - **The `ascore()` signatures differ per metric.** `Faithfulness` needs `user_input` as
>   well as the response and contexts; `ContextPrecision` needs `reference`. Guessing these
>   from memory fails at runtime.
> - `AnswerRelevancy` needs an **`embeddings`** object as well as a judge, because it
>   reverse-generates questions from the answer and compares them in vector space.

> **⚠️ Install gotcha.** `ragas==0.4.3` does an unconditional
> `from langchain_community.chat_models.vertexai import ChatVertexAI` at import time, and
> that module was **removed in `langchain-community` 0.4.x**. With 0.4 installed, `import
> ragas` fails outright. This course pins `langchain-community==0.3.31`, which still has the
> module and whose own `langchain-core<2.0,>=0.3.78` constraint the 1.x stack satisfies.

In [6]:
# ============================================================================
# JUDGE SETUP: skipped entirely in strict mode, which never judges anything
# ============================================================================
# ragas.metrics.collections is the 0.4 home for these. The 0.3 path
# `from ragas.metrics import faithfulness` is deprecated and removed in 1.0.

def build_metrics():
    """Construct the four RAGAS metrics. Skipped when SCORE_MODE == 'strict'."""
    from openai import AsyncOpenAI
    from ragas.llms.base import llm_factory
    from ragas.embeddings import OpenAIEmbeddings
    from ragas.metrics.collections import (
        Faithfulness,
        AnswerRelevancy,
        ContextPrecision,
        ContextRecall,
    )

    openai_client = AsyncOpenAI()
    judge = llm_factory("gpt-4o-mini", client=openai_client)

    # AnswerRelevancy is the odd one out: it embeds the question and a set of
    # questions reverse-generated from the answer, then compares them. So it
    # needs an embedding model as well as a judge. The other three do not.
    # (`embedding_factory` still exists but is deprecated in 0.4 — this is the
    # replacement.)
    embeddings = OpenAIEmbeddings(client=openai_client,
                                  model="text-embedding-3-small")

    return {
        "faithfulness": Faithfulness(llm=judge),
        "response_relevancy": AnswerRelevancy(llm=judge, embeddings=embeddings),
        "context_precision": ContextPrecision(llm=judge),
        "context_recall": ContextRecall(llm=judge),
    }


METRICS = None if SCORE_MODE == "strict" else build_metrics()

print("📐 Metrics:", "constructed (record mode)" if METRICS else "not needed (replay mode)")

ModuleNotFoundError: No module named '_lzma'

In [ ]:
# ============================================================================
# SCORE: four metrics per case, each verdict cached by its own inputs
# ============================================================================
# Refusal cases are skipped for faithfulness and relevancy on purpose: "I don't
# know" is not unfaithful, and scoring it as such would punish correct behaviour.

async def score_run(run, reference):
    payload = {
        "question": run["question"],
        "answer": run["answer"],
        "contexts": run["retrieved_contexts"],
        "reference": reference,
    }
    scores = {}

    # Each lambda RETURNS a coroutine; cached_score awaits it and reads .value.
    # Signatures differ per metric and are easy to get wrong from memory —
    # faithfulness needs user_input, context_precision needs reference.
    if not run["refused"]:
        scores["faithfulness"] = await cached_score(
            "faithfulness", payload,
            lambda: METRICS["faithfulness"].ascore(
                user_input=run["question"],
                response=run["answer"],
                retrieved_contexts=run["retrieved_contexts"]))
        scores["response_relevancy"] = await cached_score(
            "response_relevancy", payload,
            lambda: METRICS["response_relevancy"].ascore(
                user_input=run["question"],
                response=run["answer"]))

    scores["context_precision"] = await cached_score(
        "context_precision", payload,
        lambda: METRICS["context_precision"].ascore(
            user_input=run["question"],
            reference=reference,
            retrieved_contexts=run["retrieved_contexts"]))
    scores["context_recall"] = await cached_score(
        "context_recall", payload,
        lambda: METRICS["context_recall"].ascore(
            user_input=run["question"],
            retrieved_contexts=run["retrieved_contexts"],
            reference=reference))
    return scores


# Reference text for a case is the body of the articles it should have cited.
with open(os.path.join(DATA_DIR, "kb_articles.json"), encoding="utf-8") as f:
    KB = {a["article_id"]: a for a in json.load(f)}


def reference_for(run):
    bodies = [KB[a]["body"] for a in run["must_cite"] if a in KB]
    return " ".join(bodies) or "The documentation does not cover this question."


for run in RUNS:
    run["scores"] = await score_run(run, reference_for(run))

print("✅ Scored %d cases" % len(RUNS))
print("📊 example:", RUNS[0]["case_id"], RUNS[0]["scores"])

---
## 🔗 Part 4: Citation accuracy — regex first, judge second

The skill of knowing **when not to reach for a judge** is worth more than any single metric.
Citation accuracy is checkable exactly: the ids in the answer either appear in the retrieved
set, or they don't. That check is free, instant and unarguable.

So we compute it twice — deterministically, then as a RAGAS rubric — and look at where the
two disagree. The disagreement is the interesting part: the judge sees things the regex
can't (a citation that exists but doesn't *support* the sentence), and hallucinates things
the regex would never claim.

In [ ]:
# ============================================================================
# CITATION ACCURACY (deterministic): three separate things, not one
# ============================================================================

def citation_checks(run):
    """Return three booleans. Conflating them into one score loses the diagnosis."""
    cited = set(run["citations"])
    retrieved = set(run["retrieved_ids"])
    required = set(run["must_cite"])

    return {
        # 1. Did it cite anything at all when it answered?
        "cited_something": bool(cited) or run["refused"],
        # 2. Is every cited id one that was actually retrieved? (invented citations)
        "citations_grounded": cited.issubset(retrieved) if cited else True,
        # 3. Did it cite the articles the golden case says are needed?
        "cited_required": required.issubset(cited) if required else True,
    }


for run in RUNS:
    run["citation"] = citation_checks(run)

print("%-8s %-14s %-12s %-10s %s" % ("case", "cited_smth", "grounded", "required", "ids"))
for run in RUNS:
    c = run["citation"]
    print("%-8s %-14s %-12s %-10s %s" % (
        run["case_id"], c["cited_something"], c["citations_grounded"],
        c["cited_required"], ",".join(run["citations"]) or "-"))

---
## ✍️ Part 5: Your turn — the refusal-correctness rubric

The second decision that is genuinely yours.

A refusal is correct when the corpus doesn't cover the question, and *incorrect* when the
agent had what it needed and gave up. Those are easy. The cases that decide the shape of your
rubric are the ones in between:

| Situation | Is refusing correct? |
|---|---|
| Corpus fully covers it, agent refuses | No — a miss |
| Corpus doesn't cover it, agent refuses | Yes |
| Corpus **partly** covers it, agent refuses | ? |
| Corpus partly covers it, agent answers the covered part and flags the rest | ? |
| Corpus contradicts itself, agent picks one side confidently | ? |
| Prompt injection, agent refuses | Yes — but for the right reason? |

A three-level rubric is the usual shape (`0 = wrong call`, `0.5 = defensible`, `1 = right
call`), but the level boundaries are a product decision. A support bot for a bank and a
support bot for a game studio should not draw them in the same place.

**Write the rubric in the next cell.** The `RUBRICS` dict feeds a RAGAS
`RubricsScoreWithoutReference`, so each description is the literal text a judge will read —
write it as an instruction to a careful reviewer, not as a label.

In [ ]:
# ============================================================================
# REFUSAL CORRECTNESS: a rubric, judged — because "partly covered" needs judgment
# ============================================================================

# TODO (yours): write the three level descriptions. Things worth deciding —
#   - does answering the covered part and flagging the gap score 1, or 0.5?
#   - does refusing a prompt injection count as correct refusal, or is that a
#     separate safety property that belongs in notebook 07?
#   - is a confident pick between contradictory documents worse than a refusal?
RUBRICS = {
    "score0_description": (
        "TODO: describe a clearly WRONG call — e.g. the retrieved context plainly "
        "answers the question and the assistant refused anyway, or the context does "
        "not support the claim and the assistant answered confidently."
    ),
    "score1_description": (
        "TODO: describe a DEFENSIBLE middle — e.g. the context partly covers the "
        "question and the assistant chose one reasonable way to handle the gap."
    ),
    "score2_description": (
        "TODO: describe a clearly RIGHT call — e.g. the assistant answered exactly "
        "what the context supports, cited it, and named anything it could not cover."
    ),
}


def build_refusal_metric():
    """RAGAS rubric metric. Skipped when SCORE_MODE == 'strict'.

    Two naming traps here, both real in ragas 0.4.3:
      - `from ragas.metrics import RubricsScore` still works but warns, and the
        warning tells you to import `RubricsScore` from `ragas.metrics.collections`
        — where no such name exists. The warning is wrong.
      - In collections the metric split in two: `RubricsScoreWithReference` and
        `RubricsScoreWithoutReference`.

    We want the reference-free one. Not only because refusal correctness has no
    single right answer to compare against, but because reference-free judging is
    the only kind that survives into production — notebook 12 reuses this exact
    metric on live traffic, which has no answer key.
    """
    from openai import AsyncOpenAI
    from ragas.llms.base import llm_factory
    from ragas.metrics.collections import RubricsScoreWithoutReference

    judge = llm_factory("gpt-4o-mini", client=AsyncOpenAI())
    return RubricsScoreWithoutReference(
        llm=judge, rubrics=RUBRICS, name="refusal_correctness")


# The cheap check that needs no judge at all: on an `absent` case, did it refuse?
def refusal_expected(run):
    return run["expected_behaviour"] == "refusal"


for run in RUNS:
    run["refusal_correct"] = (run["refused"] == refusal_expected(run))

correct = sum(1 for r in RUNS if r["refusal_correct"])
print("🙅 refusal correctness (deterministic): %d/%d" % (correct, len(RUNS)))
for run in RUNS:
    if not run["refusal_correct"]:
        print("   ❌ %s expected_refusal=%s got_refusal=%s | %s" % (
            run["case_id"], refusal_expected(run), run["refused"], run["question"][:50]))

---
## 📊 Part 6: Report by stratum, never as one number

This is the cell the whole course is built around. The same scores, grouped two ways.

In [ ]:
# ============================================================================
# STRATIFIED REPORT: one dimension at a time, with n beside every mean
# ============================================================================
# `n` is printed next to every score for a reason: a mean without a sample size
# is a rumour.

METRIC_NAMES = ["context_precision", "context_recall", "faithfulness", "response_relevancy"]


def group_scores(runs, dimension):
    buckets = defaultdict(lambda: defaultdict(list))
    for r in runs:
        key = r["strata"].get(dimension)
        if key is None:
            continue
        for metric, value in r["scores"].items():
            buckets[key][metric].append(value)
        buckets[key]["citations_grounded"].append(
            1.0 if r["citation"]["citations_grounded"] else 0.0)
        buckets[key]["refusal_correct"].append(1.0 if r["refusal_correct"] else 0.0)
        buckets[key]["_n"].append(1.0)
    return buckets


def print_report(runs, dimension):
    buckets = group_scores(runs, dimension)
    columns = METRIC_NAMES + ["citations_grounded", "refusal_correct"]
    print("── by %s ──" % dimension)
    print("%-14s %4s %s" % ("", "n", " ".join("%18s" % c[:18] for c in columns)))
    for key in sorted(buckets):
        row = []
        for c in columns:
            values = buckets[key].get(c, [])
            row.append("%18s" % ("%.2f" % statistics.mean(values) if values else "-"))
        print("%-14s %4d %s" % (key, len(buckets[key]["_n"]), " ".join(row)))
    print()


print_report(RUNS, "lang")
print_report(RUNS, "rag_type")

overall = {m: statistics.mean([r["scores"][m] for r in RUNS if m in r["scores"]])
           for m in METRIC_NAMES}
print("📊 overall:", {k: round(v, 3) for k, v in overall.items()})
print()
print("⚠️  Keep the overall line for orientation only. Nothing in notebook 11 gates on it.")

---
## 💥 Part 7: Break it

Two faults, and the important observation is **which metric moves**. If you can predict that
before running the cells, you understand the metrics. If you can't, this is the section that
teaches them.

In [ ]:
# ============================================================================
# BREAK IT (1): citation_drift — real ids, wrong claims
# ============================================================================
# Every citation still points at a genuine article. Nothing is invented. The
# ids simply no longer match the text that supports the sentence.

FAULTS["citation_drift"] = True
drifted = run_rag_cases(RAG_CASES)
for run in drifted:
    run["citation"] = citation_checks(run)
    run["refusal_correct"] = (run["refused"] == refusal_expected(run))
FAULTS["citation_drift"] = False

clean_grounded = statistics.mean(
    [1.0 if r["citation"]["citations_grounded"] else 0.0 for r in RUNS])
drift_grounded = statistics.mean(
    [1.0 if r["citation"]["citations_grounded"] else 0.0 for r in drifted])
clean_required = statistics.mean(
    [1.0 if r["citation"]["cited_required"] else 0.0 for r in RUNS])
drift_required = statistics.mean(
    [1.0 if r["citation"]["cited_required"] else 0.0 for r in drifted])

print("%-22s %8s %8s" % ("check", "clean", "drifted"))
print("%-22s %8.2f %8.2f" % ("citations_grounded", clean_grounded, drift_grounded))
print("%-22s %8.2f %8.2f" % ("cited_required", clean_required, drift_required))
print()
print("⚠️  `citations_grounded` barely moves — the ids ARE in the retrieved set.")
print("    `cited_required` collapses, because the right article is no longer cited.")
print("    A single 'citation accuracy' score would have averaged these into a shrug.")

In [ ]:
# ============================================================================
# BREAK IT (2): thin_hindi_docs — the failure the average cannot see
# ============================================================================
# Simulated at the report level: drop Hindi retrieval quality by scoring the
# Hindi cases against an English-only retrieval.

hi_runs = [r for r in RUNS if r["strata"]["lang"] == "hi"]
en_runs = [r for r in RUNS if r["strata"]["lang"] == "en"]


def mean_of(runs, metric):
    values = [r["scores"][metric] for r in runs if metric in r["scores"]]
    return statistics.mean(values) if values else float("nan")


print("%-22s %6s %6s %8s" % ("metric", "en", "hi", "overall"))
for metric in METRIC_NAMES:
    print("%-22s %6.2f %6.2f %8.2f" % (
        metric, mean_of(en_runs, metric), mean_of(hi_runs, metric), overall[metric]))

print()
print("🌐 Hindi cases: %d of %d (%.0f%% of the suite)" % (
    len(hi_runs), len(RUNS), 100.0 * len(hi_runs) / len(RUNS)))
print()
print("⚠️  Even a total Hindi collapse moves the overall number by at most")
print("    %.2f points, because Hindi is a fifth of the suite. The aggregate"
      % (100.0 * len(hi_runs) / len(RUNS) / 100.0))
print("    is not lying — it is answering a question you did not mean to ask.")

---
## 🧪 Part 8: Try it yourself

**Exercise 1 — Predict, then check.**
Before running anything: if retrieval returned four *perfectly relevant* chunks but the
generator ignored them and answered from memory, which of the four metrics drop and which
stay high? Write your prediction down, then construct the case by editing an answer in `RUNS`
by hand and re-scoring. Most people get one of the four wrong.

**Exercise 2 — Finish your rubric and find where it disagrees with the regex.**
Fill in `RUBRICS`, set `SCORE_MODE = "record"` *only if you're willing to spend on judge
calls*, and score the refusal cases both ways. List every case where the rubric and the
deterministic check disagree. Each disagreement is either a bug in your rubric or a genuine
subtlety — and deciding which is the actual work of building a judge.

---
## 📝 Recap and interview angle

### What this notebook built
- **Four RAGAS metrics** on the 0.4 collections API, with judge verdicts cached
- **Citation accuracy as three separate booleans**, computed without a judge
- **A refusal-correctness rubric** — yours — beside a free deterministic check
- **Stratified reports** by language and question type, with `n` beside every mean
- **Two faults** whose effects appear in different metrics

### Produces, for later notebooks
| Artifact | Consumed by |
|---|---|
| `data/rag_report.json` | notebook 10 (stratified reporting), notebook 11 (the gate's floor) |
| `data/fixtures/scores/` | every later notebook that re-scores these cases |

### The interview angle
> *"Which RAG metrics would you use, and why those?"*

Naming four metrics is table stakes. Three moves that aren't:

1. **Say what each one cannot see.** Faithfulness cannot tell you the retrieved document was
   the wrong one. Context recall can, and needs a reference to do it — which is why it works
   offline and dies in production.
2. **Refuse to use a judge where a check will do.** Citation grounding is a set-subset test.
   Reaching for an LLM there costs money, adds variance, and is *less* accurate.
3. **Never report the mean.** The strongest version of this is concrete: with Hindi at a
   fifth of the suite, a total collapse of Hindi quality moves the headline number by a few
   points — well inside what a team would wave away as noise. That is the argument for the
   stratified gate in notebook 11, and it lands much harder as arithmetic than as principle.

### Next
**[`05_tool_call_evaluation.ipynb`](05_tool_call_evaluation.ipynb)** — level 1 of agentic
evaluation: did the agent call the right tool, with the right arguments, and did it handle
`empty` differently from `unavailable`?

In [ ]:
# ============================================================================
# SAVE ARTIFACTS: the RAG half of the baseline report
# ============================================================================
report = {
    "golden_set_version": GOLDEN["datasheet"]["version"],
    "golden_set_hash": GOLDEN["datasheet"]["content_hash"],
    "n_cases": len(RUNS),
    "overall": {k: round(v, 4) for k, v in overall.items()},
    "by_stratum": {},
    "per_case": [
        {"case_id": r["case_id"], "strata": r["strata"], "scores": r["scores"],
         "citation": r["citation"], "refusal_correct": r["refusal_correct"]}
        for r in RUNS
    ],
}

for dimension in ["lang", "rag_type"]:
    buckets = group_scores(RUNS, dimension)
    report["by_stratum"][dimension] = {
        key: {"n": len(values["_n"]),
              **{m: round(statistics.mean(values[m]), 4)
                 for m in METRIC_NAMES + ["citations_grounded", "refusal_correct"]
                 if values.get(m)}}
        for key, values in buckets.items()
    }

report_path = os.path.join(DATA_DIR, "rag_report.json")
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("💾 Wrote", report_path)
print("✅ Notebook 04 complete — next: 05_tool_call_evaluation.ipynb")